# Slotera — ML Exploration Notebook

Interactive exploration of reservation data for feature engineering and model prototyping.

## Setup
Make sure the Postgres database is running (`docker compose up -d` from `server/`)  
and that seed data has been loaded (`SEED_DATA=true python -m db.migrate`).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print('Setup complete')

## 1. Load Raw Data

In [ ]:
from src.db import load_reservations, load_customers, load_businesses, load_service_types

reservations = load_reservations()
customers = load_customers()
businesses = load_businesses()
service_types = load_service_types()

print(f'Reservations: {len(reservations)}')
print(f'Customers:    {len(customers)}')
print(f'Businesses:   {len(businesses)}')
print(f'Services:     {len(service_types)}')

reservations.head()

## 2. Feature Engineering

In [ ]:
from src.features.reservation_features import build_reservation_features, build_daily_demand
from src.features.customer_features import build_customer_features, build_rfm_features

# Reservation-level features
res_features = build_reservation_features(reservations)
print('Reservation features:')
res_features[['reservation_id', 'business_name', 'status', 'day_of_week', 
               'hour_of_day', 'is_weekend', 'lead_time_hours', 'guest_capacity_ratio',
               'has_payment', 'is_cancelled']].head(10)

In [ ]:
# Customer-level features (RFM + behavioral)
cust_features = build_customer_features(reservations, customers)
print('Customer features:')
cust_features[['customer_id', 'name', 'total_reservations', 'total_spend',
                'recency_days', 'cancellation_rate', 'unique_businesses']].head()

In [ ]:
# RFM matrix
rfm = build_rfm_features(cust_features)
print('RFM features:')
rfm

## 3. Customer Segmentation

In [ ]:
from src.models.segmentation import CustomerSegmentationModel

seg_model = CustomerSegmentationModel(n_clusters=min(4, len(rfm)))
segments = seg_model.fit_predict(rfm)

print('Segment distribution:')
print(segments['segment_label'].value_counts())
print()
segments

## 4. Cancellation Prediction

In [ ]:
from src.models.cancellation import CancellationPredictionModel

cancel_model = CancellationPredictionModel()
cancel_df = cancel_model.prepare_features(res_features, cust_features)

print(f'Training samples: {len(cancel_df)}')
print(f'Cancelled: {cancel_df["is_cancelled"].sum()}')
print(f'Completed: {(~cancel_df["is_cancelled"].astype(bool)).sum()}')

if len(cancel_df) >= 10:
    result = cancel_model.train_and_evaluate(cancel_df)
    print(f'\nAUC-ROC: {result.metrics.auc_roc:.3f}')
    print(f'F1:      {result.metrics.f1:.3f}')
    print(f'\nFeature importance:')
    for feat, imp in list(result.feature_importance.items())[:10]:
        print(f'  {feat}: {imp}')
else:
    print('Not enough samples for training')

## 5. Demand Forecasting

In [ ]:
from src.features.reservation_features import build_demand_timeseries_features
from src.models.demand_forecast import DemandForecastModel

daily = build_daily_demand(res_features)
demand_ts = build_demand_timeseries_features(daily)

print(f'Daily demand records: {len(demand_ts)}')
print(f'Businesses: {demand_ts["business_id"].nunique()}')

demand_model = DemandForecastModel(forecast_horizon=7)
forecast_result = demand_model.train_and_evaluate(demand_ts)

print('\nMetrics per business:')
for biz, m in forecast_result.metrics.items():
    print(f'  {biz}: MAE={m.mae:.2f}, R²={m.r2:.3f}')

if not forecast_result.forecasts.empty:
    print('\n7-day forecasts:')
    print(forecast_result.forecasts[['business_name', 'date', 'predicted_reservations']].to_string(index=False))

## 6. Run Full Pipeline

In [ ]:
from src.pipelines.insights_pipeline import InsightsPipeline

pipeline = InsightsPipeline()
results = pipeline.run(store_results=False)  # Don't write to DB from notebook

print(f'Run ID: {results["run_id"]}')
print(f'Elapsed: {results["elapsed_seconds"]}s')
print(f'Data: {results["data"]}')
print(f'Segmentation: {results["segmentation"]["status"]}')
print(f'Cancellation: {results["cancellation"]["status"]}')
print(f'Demand:       {results["demand_forecast"]["status"]}')